# Face Recognition and Responsible Use

> **Advanced · Optional applied vision**


## Why this matters

Recognizing a person is not the same as detecting a face. This lesson treats legacy LBPH as a learning exercise and centers evaluation, consent, privacy, and limits.

**Where it appears:** Small, consented closed-set demonstrations and understanding legacy OpenCV recognition pipelines—not unattended real-world identification.


## Learning Objectives

- Understand the embedding-based face recognition pipeline (detect -> align -> embed -> compare)
- Use OpenCV's built-in LBPH recognizer as a lightweight, fully-offline baseline
- Evaluate recognition with a proper train/test split, not just training accuracy


## Prerequisites

17 Face Detection and Landmarks

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

LBPH recognizer, train/test split, distance thresholds, false acceptance and rejection

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Face Recognition

Modern face recognition converts an aligned face crop into a fixed-length
embedding vector (via a trained deep network) such that same-identity
faces are close in embedding space and different identities are far apart
(measured by e.g. cosine or Euclidean distance) -- this is the same
paradigm used by FaceNet/ArcFace-style systems. OpenCV's `cv2.face`
module includes a simpler, fully classical, no-download alternative:
**LBPH** (Local Binary Patterns Histograms), which this notebook uses so
the pipeline runs entirely offline.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 1. Building a small labeled face dataset

Generate several synthetic 'identities' with per-identity variation (each face shape slightly different) so recognition has a genuine (if simplified) problem to solve.


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from cv_utils import load_real_image, get_real_data, show_grid, has_module


def load_dataset_faces() -> dict:
    """Load real face images from the dataset and group by identity."""
    data_dir = get_real_data("images/faces", "dataset")
    identities = {}
    for i, subj_dir in enumerate(sorted(data_dir.iterdir())):
        if not subj_dir.is_dir():
            continue
        faces = []
        for img_path in sorted(subj_dir.glob("*.pgm")):
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            faces.append(img)
        identities[i] = faces
    return identities


identities = load_dataset_faces()
show_grid([(f"identity {i}, sample 0", faces[0]) for i, faces in identities.items()])

### 2. Training LBPH and evaluating on held-out samples

Split each identity's samples into train/test, train `cv2.face.LBPHFaceRecognizer`, and report accuracy on the *held-out* samples -- not training accuracy, which would be misleading.


In [ ]:
def train_test_split_faces(identities: dict, n_test: int = 2):
    train_images, train_labels, test_images, test_labels = [], [], [], []
    for label, faces in identities.items():
        train_images += faces[:-n_test]
        train_labels += [label] * (len(faces) - n_test)
        test_images += faces[-n_test:]
        test_labels += [label] * n_test
    return train_images, np.array(train_labels), test_images, np.array(test_labels)


train_imgs, train_labels, test_imgs, test_labels = train_test_split_faces(identities)

if not hasattr(cv2, "face"):
    print(
        "cv2.face requires opencv-contrib-python (pip install opencv-contrib-python)."
    )
    print(
        "Skipping face recognition training... please switch to an environment with opencv-contrib-python to run this."
    )
else:
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.train(train_imgs, train_labels)

    correct = 0
    for img, true_label in zip(test_imgs, test_labels):
        predicted_label, confidence = recognizer.predict(img)
        correct += int(predicted_label == true_label)
        print(
            f"true={true_label}  predicted={predicted_label}  confidence(lower=better)={confidence:.1f}"
        )

    print(f"\nHeld-out accuracy: {correct}/{len(test_imgs)}")

### 3. Setting a rejection threshold for unknown faces

A real system must reject faces that don't match any known identity confidently -- LBPH's `confidence` (a distance, lower=better) is used as that threshold.


In [ ]:
def identify_or_reject(
    recognizer, face_img: np.ndarray, max_confidence: float = 70.0
) -> str:
    label, confidence = recognizer.predict(face_img)
    if confidence > max_confidence:
        return f"unknown (best guess was identity {label}, confidence {confidence:.1f} too high)"
    return f"identity {label} (confidence {confidence:.1f})"


if hasattr(cv2, "face"):
    # Load an image not in the training set
    unknown_img = load_real_image("images/faces", "yoga.jpg")
    unknown_face = cv2.cvtColor(unknown_img, cv2.COLOR_BGR2GRAY)
    unknown_face = cv2.resize(
        unknown_face, (test_imgs[0].shape[1], test_imgs[0].shape[0])
    )
    print(identify_or_reject(recognizer, unknown_face))
    print(identify_or_reject(recognizer, test_imgs[0]))

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Face Recognition: PCA and Distance Metrics for Feature Similarity

Modern face recognition matches vectors (embeddings) generated from faces. Here, we simulate this process by using Principal Component Analysis (PCA) to downsample facial images to feature vectors, and match them using Cosine Similarity.


In [ ]:
from sklearn.decomposition import PCA

# Create synthetic dataset: 10 mock face vector arrays (shape: 10 x 400 pixels)
rng = np.random.default_rng(42)
faces = rng.normal(128, 30, (10, 400))

# Apply Principal Component Analysis (eigenfaces)
pca = PCA(n_components=5)
embeddings = pca.fit_transform(faces)

# Match a query face
query_face = faces[0] + rng.normal(0, 5, 400)  # Query is similar to index 0
query_emb = pca.transform(query_face.reshape(1, -1))[0]

# Compute Cosine Similarity distance
similarities = []
for emb in embeddings:
    dot = np.dot(query_emb, emb)
    norm_q = np.linalg.norm(query_emb)
    norm_e = np.linalg.norm(emb)
    similarity = dot / (norm_q * norm_e)
    similarities.append(similarity)

best_idx = np.argmax(similarities)
print("Cosine similarity scores:", [f"{s:.3f}" for s in similarities])
print(
    f"Query matched target index: {best_idx} (Similarity score: {similarities[best_idx]:.4f})"
)

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Face Recognition
1. Increase `n_samples` per identity and re-measure held-out accuracy -- more data should help LBPH.
2. Sweep the rejection `max_confidence` threshold and describe the false-accept vs false-reject trade-off.
3. Research (markdown-only) how a modern embedding-based system like ArcFace differs from LBPH's histogram-comparison approach.

Use the empty cell below to work through them.


#### Solutions — Face Recognition

In [ ]:
# Solution 1: LBPH accuracy vs sample count
# Explanation: Increasing the sample count `n_samples` from 5 to 20 per identity gives
# the LBPH recognizer a wider range of pose and lighting variations. This results in
# improved recognition accuracy on unseen validation frames, reducing classification error.

if hasattr(cv2, "face"):
    print("Testing with VERY FEW training samples (2 per identity):")
    # First, get a standard split with 2 test samples and 8 train samples per identity
    t_imgs_full, t_lbls_full, test_i, test_l = train_test_split_faces(
        identities, n_test=2
    )

    # We only take the first 2 training samples per identity for the 'few' model
    t_imgs_few, t_lbls_few = [], []
    for label in set(t_lbls_full):
        indices = [i for i, l in enumerate(t_lbls_full) if l == label][:2]
        t_imgs_few.extend([t_imgs_full[i] for i in indices])
        t_lbls_few.extend([t_lbls_full[i] for i in indices])

    rec_few = cv2.face.LBPHFaceRecognizer_create()
    rec_few.train(t_imgs_few, np.array(t_lbls_few))
    corr_few = sum(
        1 for img, lbl in zip(test_i, test_l) if rec_few.predict(img)[0] == lbl
    )
    print(
        f"Accuracy with 2 samples: {corr_few}/{len(test_i)} ({corr_few / len(test_i) * 100:.1f}%)\n"
    )

    print("Testing with MORE training samples (8 per identity):")
    rec_more = cv2.face.LBPHFaceRecognizer_create()
    rec_more.train(t_imgs_full, t_lbls_full)
    corr_more = sum(
        1 for img, lbl in zip(test_i, test_l) if rec_more.predict(img)[0] == lbl
    )
    print(
        f"Accuracy with 8 samples: {corr_more}/{len(test_i)} ({corr_more / len(test_i) * 100:.1f}%)\n"
    )

In [ ]:
# Solution 2: sweep max_confidence threshold trade-offs
# Explanation: The confidence score returned by LBPH is actually a distance metric (lower is better).
# Setting a strict low threshold (e.g. 50) minimizes False Acceptances (unknowns registered as users)
# but increases False Rejections (valid users rejected). Setting a loose threshold (e.g. 150)
# does the inverse.

if hasattr(cv2, "face"):
    print("Sweeping thresholds on an Unknown face and a Known face:")
    for thresh in [30.0, 70.0, 150.0]:
        print(f"\n--- Threshold: {thresh} ---")
        res_unknown = identify_or_reject(
            recognizer, unknown_face, max_confidence=thresh
        )
        res_known = identify_or_reject(recognizer, test_imgs[0], max_confidence=thresh)
        print(f"Unknown Face -> {res_unknown}")
        print(f"Known Face -> {res_known}")

In [ ]:
# Solution 3: ArcFace vs LBPH representation difference
# LBPH computes localized histograms of texture changes directly from the pixel grid, ignoring
# high-level facial layout structures and failing under rotation. ArcFace uses deep convolutional
# networks trained on angular margin loss to generate compact, highly discriminative 512-dimensional
# feature vectors (embeddings) that remain robust to scale, rotation, and lighting variations.

import urllib.request
from pathlib import Path


def demo_sface():
    # For this demo, we simulate the SFace embedding extraction API conceptually,
    # as downloading the full 30MB ONNX models takes time. In production,
    # you would use cv2.FaceRecognizerSF.create(...)
    print("--- Conceptual SFace API Usage ---")
    print(
        "1. Load deep learning ONNX models for detection (YuNet) and recognition (SFace)"
    )
    print(
        'detector = cv2.FaceDetectorYN.create("face_detection_yunet.onnx", "", (320, 320))'
    )
    print('recognizer = cv2.FaceRecognizerSF.create("face_recognizer_fast.onnx", "")')
    print("\n2. Extract embeddings")
    print("_, faces = detector.detect(img)")
    print("aligned_face = recognizer.alignCrop(img, faces[0])")
    print("feature = recognizer.feature(aligned_face)")
    print("\n3. Compare embeddings with Cosine Similarity")
    print(
        "score = recognizer.match(feature1, feature2, cv2.FaceRecognizerSF_FR_COSINE)"
    )
    print('if score > 0.363: print("Same Identity")')


demo_sface()

## Summary

You can build and evaluate a small closed-set baseline, set a rejection threshold, and articulate why this is not a deployment-ready recognition system.

- **Best Practices:** Use only consented data, retain it for the minimum time, measure false matches, include an unknown class, and document intended use and exclusions.
- **Common Pitfalls:** Using a tiny convenience dataset as evidence of accuracy, forcing an identity on low confidence, and deploying biometric recognition without governance.